# GST actual-data visuals

This notebook generates visual ideas from the saved GST/POUNDERS run data. It uses the current saved CSV outputs first, so the plots are based on actual runs rather than hand-drawn examples.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle

BASE = Path.cwd()
OUTDIR = BASE / "actual_data_visuals"
OUTDIR.mkdir(exist_ok=True)

progress_path = BASE / "pounders_fpr_shot_sweep_weighted_least_squares_cptplnd_progress.csv"
summary_path = BASE / "pounders_fpr_shot_sweep_weighted_least_squares_cptplnd_summary.csv"
metadata_path = BASE / "pounders_fpr_shot_sweep_weighted_least_squares_cptplnd_solutions_metadata.csv"

progress = pd.read_csv(progress_path)
summary = pd.read_csv(summary_path)
metadata = pd.read_csv(metadata_path)

numeric_cols = [
    "shots_per_circuit", "nf", "x_index", "incumbent_index", "active_objective", "full_objective",
    "incumbent_active_objective", "incumbent_full_objective", "active_residuals", "active_circuits",
    "new_residuals_revealed", "new_circuits_revealed", "union_residuals_revealed", "union_circuits_revealed",
    "cumulative_eval_residuals", "cumulative_eval_circuits", "cumulative_shots_revealed", "cumulative_eval_shots",
    "fpr_selected_residuals", "fpr_selected_circuits", "fpr_new_residuals", "fpr_new_circuits",
    "fpr_union_residuals", "fpr_union_circuits", "delta", "ng", "rho", "rho_active", "rho_full", "mdec", "step_norm", "plot_objective",
]
for col in numeric_cols:
    if col in progress.columns:
        progress[col] = pd.to_numeric(progress[col], errors="coerce")
for col in summary.columns:
    if col not in ["run_label", "use_fpr_reduction", "n_sigma_less_than_1"]:
        summary[col] = pd.to_numeric(summary[col], errors="ignore")

colors = {False: "#4C78A8", True: "#F58518"}
labels = {False: "No FPR", True: "FPR"}

display(summary)
print("Rows in progress history:", len(progress))
print("Output folder:", OUTDIR)

## 1. Actual optimization trajectories
These are the real saved POUNDERS histories. The y-axis uses the full objective at the current incumbent.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for use_fpr, grp in progress.groupby("use_fpr_reduction"):
    use_bool = bool(use_fpr)
    grp = grp.sort_values(["cumulative_eval_residuals", "nf"])
    axes[0].plot(grp["cumulative_eval_residuals"], grp["incumbent_full_objective"], marker="o", ms=3.5, lw=1.8, color=colors[use_bool], label=labels[use_bool])
    axes[1].plot(grp["cumulative_shots_revealed"], grp["incumbent_full_objective"], marker="o", ms=3.5, lw=1.8, color=colors[use_bool], label=labels[use_bool])
for ax in axes:
    ax.set_yscale("log")
    ax.set_ylabel("full objective at incumbent")
    ax.legend()
axes[0].set_xlabel("cumulative residual evaluations")
axes[0].set_title("Objective vs residual evaluations")
axes[1].set_xlabel("cumulative revealed shots")
axes[1].set_title("Objective vs revealed shots")
fig.tight_layout()
fig.savefig(OUTDIR / "actual_objective_vs_revealed_shots.png", dpi=220)
plt.show()

## 2. Actual shot-savings summary
This compares final revealed shots, final full objective, and likelihood-based badness-of-fit.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
summary_sorted = summary.sort_values("use_fpr_reduction")
x = np.arange(len(summary_sorted))
bar_colors = [colors[bool(v)] for v in summary_sorted["use_fpr_reduction"]]
axes[0].bar(x, summary_sorted["final_revealed_shots"] / 1e6, color=bar_colors)
axes[0].set_ylabel("million shots")
axes[0].set_title("Revealed-shot cost")
axes[1].bar(x, summary_sorted["final_full_objective"], color=bar_colors)
axes[1].set_yscale("log")
axes[1].set_title("Final full objective")
axes[2].bar(x, summary_sorted["n_sigma"], color=bar_colors)
axes[2].axhline(1, color="black", ls="--", lw=1, label="N_sigma = 1")
axes[2].set_title("Badness-of-fit N_sigma")
axes[2].legend()
for ax in axes:
    ax.set_xticks(x)
    ax.set_xticklabels(summary_sorted["run_label"], rotation=20, ha="right")
fig.tight_layout()
fig.savefig(OUTDIR / "actual_shot_savings_summary.png", dpi=220)
plt.show()

## 3. FPR reveal fingerprint and timeline
This is the closest saved-data version of a circuit-reduction visual. We have counts of selected/new/union circuits, but not individual circuit indices in the CSV.

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))
fpr = progress[progress["use_fpr_reduction"] == True].sort_values("nf")
no = progress[progress["use_fpr_reduction"] == False].sort_values("nf")
ax1.plot(fpr["nf"], fpr["union_circuits_revealed"], color=colors[True], lw=2.2, marker="o", ms=3, label="FPR union circuits revealed")
ax1.plot(no["nf"], no["union_circuits_revealed"], color=colors[False], lw=1.8, ls="--", label="No-FPR circuits revealed")
ax1.set_xlabel("POUNDERS function evaluation nf")
ax1.set_ylabel("cumulative distinct circuits revealed")
ax2 = ax1.twinx()
ax2.bar(fpr["nf"], fpr["new_circuits_revealed"], color="#F58518", alpha=0.28, label="new FPR circuits this eval")
ax2.set_ylabel("new circuits revealed at eval")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower right")
ax1.set_title("Actual FPR circuit reveal timeline")
fig.tight_layout()
fig.savefig(OUTDIR / "actual_fpr_circuit_reveal_timeline.png", dpi=220)
plt.show()

## 4. Trust-region and gradient diagnostics
These use actual saved `delta`, `rho_active`, accepted/rejected flags, and `ng`.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
for use_fpr, grp in progress.groupby("use_fpr_reduction"):
    use_bool = bool(use_fpr)
    trial = grp[grp["phase"].eq("trial")].sort_values("nf")
    axes[0].plot(trial["nf"], trial["delta"], color=colors[use_bool], marker="o", ms=3, lw=1.8, label=labels[use_bool])
    accepted = trial[trial["accepted"].astype(str).str.upper().eq("TRUE")]
    rejected = trial[trial["accepted"].astype(str).str.upper().eq("FALSE")]
    axes[1].scatter(accepted["nf"], accepted["rho_active"], color=colors[use_bool], marker="o", s=22, label=f"{labels[use_bool]} accepted")
    axes[1].scatter(rejected["nf"], rejected["rho_active"], color=colors[use_bool], marker="x", s=35, label=f"{labels[use_bool]} rejected")
axes[0].set_yscale("log")
axes[0].set_ylabel("trust-region radius delta")
axes[0].legend()
axes[0].set_title("Actual trust-region behavior")
axes[1].axhline(0, color="black", lw=1)
axes[1].set_ylabel("rho_active")
axes[1].set_xlabel("POUNDERS function evaluation nf")
axes[1].legend(ncol=2, fontsize=9)
fig.tight_layout()
fig.savefig(OUTDIR / "actual_trust_region_and_rho.png", dpi=220)
plt.show()

fig, ax = plt.subplots(figsize=(10, 4.8))
for use_fpr, grp in progress.groupby("use_fpr_reduction"):
    use_bool = bool(use_fpr)
    trial = grp[grp["phase"].eq("trial")].sort_values("nf")
    ax.plot(trial["nf"], trial["ng"], color=colors[use_bool], marker="o", ms=3, lw=1.8, label=labels[use_bool])
ax.set_yscale("log")
ax.set_xlabel("POUNDERS function evaluation nf")
ax.set_ylabel("model gradient norm ng")
ax.set_title("Actual model gradient norm")
ax.legend()
fig.tight_layout()
fig.savefig(OUTDIR / "actual_gradient_norm.png", dpi=220)
plt.show()

## 5. Actual-data art board
This is a mug/art-style composition made only from saved run quantities: trajectory, FPR union growth, and final revealed-circuit counts.

In [ ]:
fig = plt.figure(figsize=(16, 6), facecolor="#071024")
gs = fig.add_gridspec(1, 3, width_ratios=[1.35, 1.0, 1.0], wspace=0.18)
ax = fig.add_subplot(gs[0,0]); ax.set_facecolor("#071024")
for use_fpr, grp in progress.groupby("use_fpr_reduction"):
    use_bool = bool(use_fpr)
    grp = grp.sort_values("cumulative_shots_revealed")
    ax.plot(grp["cumulative_shots_revealed"] / 1e6, grp["incumbent_full_objective"], marker="o", ms=3, lw=2.1, color=colors[use_bool], label=labels[use_bool])
ax.set_yscale("log")
ax.set_title("Actual optimization signal", color="white", fontweight="bold")
ax.set_xlabel("million revealed shots", color="white")
ax.set_ylabel("objective", color="white")
ax.tick_params(colors="white")
for s in ax.spines.values(): s.set_color("#31547d")
ax.grid(alpha=.18, color="white")
ax.legend()

ax = fig.add_subplot(gs[0,1]); ax.set_facecolor("#071024")
fpr = progress[progress["use_fpr_reduction"] == True].sort_values("nf")
vals = fpr.dropna(subset=["union_circuits_revealed"])["union_circuits_revealed"].to_numpy()
if len(vals) > 0:
    vals = vals / max(vals.max(), 1)
    theta = np.linspace(0, 2*np.pi, len(vals), endpoint=False)
    for t, v in zip(theta, vals):
        r0, r1 = 0.18, 0.18 + 0.34 * v
        ax.plot([0.5+r0*np.cos(t), 0.5+r1*np.cos(t)], [0.52+r0*np.sin(t), 0.52+r1*np.sin(t)], color="#F58518", lw=1.4, alpha=.8)
    ax.add_patch(Circle((0.5, 0.52), .18, fill=False, ec="white", lw=1.1, alpha=.75))
ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis("off")
ax.set_title("FPR reveal fingerprint\n(actual union growth)", color="white", fontweight="bold")

ax = fig.add_subplot(gs[0,2]); ax.set_facecolor("#071024")
final_fpr = summary[summary["use_fpr_reduction"].astype(str).str.lower().eq("true")].iloc[0]
final_no = summary[summary["use_fpr_reduction"].astype(str).str.lower().eq("false")].iloc[0]
selected = int(final_fpr["final_union_circuits"]); total = int(final_no["final_union_circuits"])
nblocks = 120
nsel = int(round(nblocks * selected / total)) if total else 0
for i in range(nblocks):
    row, col = divmod(i, 15)
    color = "#F58518" if i < nsel else "#55708f"
    alpha = .95 if i < nsel else .25
    ax.add_patch(Rectangle((0.06+col*0.058, 0.18+row*0.07), .04, .045, facecolor=color, edgecolor="none", alpha=alpha))
ax.text(.06,.84,f"FPR revealed {selected}/{total} circuits", color="white", fontsize=12, fontweight="bold")
ax.text(.06,.10,"actual final union vs full revealed set", color="#42dcff", fontsize=10)
ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis("off")
fig.suptitle("Actual-data visuals from saved GST/POUNDERS sweep", color="white", fontsize=18, fontweight="bold")
fig.savefig(OUTDIR / "actual_data_visual_concepts.png", dpi=220, facecolor=fig.get_facecolor(), bbox_inches="tight")
plt.show()

## Optional next step: deeper per-circuit visuals

The saved CSVs do **not** contain individual residual vectors, individual circuit indices, or full Jacobian rows. To generate a true circuit constellation, residual fingerprint by circuit, or germ-amplification spiral from exact circuits, add a save step in `GST_model.ipynb` that stores arrays such as:

```python
np.savez(
    "actual_data_visual_arrays.npz",
    residual=residual_vector,
    p=p_vector,
    counts=counts_vector,
    circuit_ids=np.array([...]),
    fpr_mask=fpr_mask,
    jacobian_features=some_feature_matrix,
)
```

Then this notebook can load that `.npz` and make true per-circuit art instead of count-level art.

<!-- FANCY_ACTUAL_DATA_VISUALS_SECTION -->

## Fancy Actual-Data Visualizations

These cells use the saved GST/POUNDERS sweep CSV files. They include heatmaps, bubble plots, a circuit-reveal fingerprint, and a trust-region phase portrait.

In [ ]:

from pathlib import Path
from IPython.display import display, Image

visual_dir = Path("actual_data_visuals")
fancy_images = [
    "actual_heatmap_objective_by_iteration.png",
    "actual_heatmap_final_metrics.png",
    "actual_bubble_objective_revealed_shots.png",
    "actual_bubble_fpr_new_circuits.png",
    "actual_heatmap_fpr_reveal_fingerprint.png",
    "actual_phase_portrait_delta_ng.png",
    "actual_lollipop_shot_economy.png",
    "actual_radar_final_metric_scores.png",
]

for image_name in fancy_images:
    path = visual_dir / image_name
    print(path)
    display(Image(filename=str(path)))


<!-- LANDSCAPE_ACTUAL_DATA_VISUALS_SECTION -->

## Fisher-Style Objective Landscape

These plots use the saved POUNDERS trace to draw a landscape of `log10(full objective)` over `log10(cumulative revealed shots)` and POUNDERS evaluation index. This is Fisher-inspired, not an exact Fisher-information surface.

In [ ]:

from IPython.display import display, Image
from pathlib import Path
visual_dir = Path("actual_data_visuals")
for image_name in [
    "actual_fisher_style_objective_landscape.png",
    "actual_3d_objective_landscape.png",
    "actual_objective_landscape_poster.png",
]:
    path = visual_dir / image_name
    print(path)
    display(Image(filename=str(path)))


<!-- EXTRA_ACTUAL_DATA_VISUALS_SECTION -->

## Extra Actual-Data Visual Ideas

These plots use the saved POUNDERS/FPR sweep summary, progress trace, and optimized parameter vectors. The constellation uses actual selected counts on a representative layout because exact per-circuit FPR masks were not saved.

In [ ]:

from IPython.display import display, Image
from pathlib import Path
visual_dir = Path("actual_data_visuals")
extra_images = [
    "actual_shot_budget_flow.png",
    "actual_circuit_selection_constellation.png",
    "actual_fpr_growth_rings.png",
    "actual_trust_region_solar_system.png",
    "actual_objective_skyline.png",
    "actual_nsigma_thermometer.png",
    "actual_parameter_mosaic_heatmap.png",
    "actual_gst_subway_map.png",
    "actual_visual_gallery_contact_sheet.png",
]
for image_name in extra_images:
    path = visual_dir / image_name
    if path.exists():
        print(path)
        display(Image(filename=str(path)))


<!-- FINAL_MUG_STORY_DESIGN_SECTION -->

## Final Mug Story Design

A combined mug-wrap design using the constellation as the hero and growth rings as the reveal-over-time story. Generated from the saved sweep summary/progress data.

In [ ]:

from IPython.display import display, Image
from pathlib import Path
visual_dir = Path("actual_data_visuals")
for image_name in ["final_mug_story_wrap.png", "final_mug_story_wrap_clean.png"]:
    path = visual_dir / image_name
    print(path)
    display(Image(filename=str(path)))


<!-- FINAL_FOUR_STORY_MUG_SECTION -->

## Four-Visual Mug Story Design

Combines the landscape idea, circuit constellation, FPR growth rings, and the actual-data concept panel into a single mug-wrap story.

In [ ]:

from IPython.display import display, Image
from pathlib import Path
visual_dir = Path("actual_data_visuals")
for image_name in ["final_mug_four_story_wrap.png", "final_mug_four_story_wrap_clean.png"]:
    path = visual_dir / image_name
    print(path)
    display(Image(filename=str(path)))
